# NDVI CDR — GeoZarr Multiscale Pyramid via topozarr → Icechunk on R2

Takes the virtual-reference Icechunk store created in `virtualizarr_ndvi_cdr_r2.ipynb`
and builds a **GeoZarr multiscale pyramid** from it using
[topozarr](https://github.com/carbonplan/topozarr), writing materialized (real) chunk
data to a separate Icechunk repository on the same R2 bucket.

| | |
|---|---|
| **Source store** | `s3://osc-pub/ndvi-cdr-icechunk` (virtual refs → AWS S3) |
| **Pyramid store** | `s3://osc-pub/ndvi-cdr-pyramid` (materialised, GeoZarr multiscales) |
| **CRS** | EPSG:4326 |
| **Levels** | 4 (full-res + 3 downscaled) |
| **Method** | mean |


## Setup

In [ ]:
import os
import warnings
from dotenv import load_dotenv

import xarray as xr
import xproj  # noqa: registers .proj accessor
import icechunk
from topozarr.coarsen import create_pyramid

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

In [ ]:
load_dotenv(f'{os.environ["HOME"]}/dotenv/r2-osc-pub.env')

r2_bucket   = "osc-pub"
r2_endpoint = os.environ["ENDPOINT_URL"]

## 1. Open the source Icechunk store (virtual references)

The store created by `virtualizarr_ndvi_cdr_r2.ipynb` holds virtual chunk references
that point back to the original files on `s3://noaa-cdr-ndvi-pds`.  We need to pass
anonymous S3 credentials so Icechunk can resolve those references at read time.

In [ ]:
src_storage = icechunk.s3_storage(
    bucket=r2_bucket,
    prefix="ndvi-cdr-icechunk",
    from_env=True,
    endpoint_url=r2_endpoint,
    region="auto",
    force_path_style=False,
)

src_config = icechunk.RepositoryConfig.default()
src_config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://noaa-cdr-ndvi-pds/",
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    ),
)

credentials = icechunk.containers_credentials({
    "s3://noaa-cdr-ndvi-pds/": icechunk.s3_credentials(anonymous=True)
})

src_repo = icechunk.Repository.open(
    src_storage, src_config, authorize_virtual_chunk_access=credentials
)
src_session = src_repo.readonly_session("main")

ds = xr.open_zarr(src_session.store, consolidated=False)
ds

## 2. Prepare the NDVI layer for pyramiding

Select only the `NDVI` variable (scale/offset already decoded by xarray) and assign the
CRS via `xproj` — required by topozarr to embed GeoZarr `proj` and `spatial` metadata.

In [ ]:
ndvi = ds[["NDVI"]].proj.assign_crs(spatial_ref="EPSG:4326")
ndvi

## 3. Build the GeoZarr multiscale pyramid

`create_pyramid` coarsens the data by 2× at each level and writes the GeoZarr
`multiscales`, `proj`, and `spatial` metadata.  `pyramid.encoding` contains
chunk/shard sizes tuned for ~500 KB web-visualization tiles.

| Level | Shape (lat × lon) | Scale factor |
|---|---|---|
| 0 | 3600 × 7200 | 1× (full res) |
| 1 | 1800 × 3600 | 2× |
| 2 | 900 × 1800 | 4× |
| 3 | 450 × 900 | 8× |

In [ ]:
pyramid = create_pyramid(
    ndvi,
    levels=4,
    x_dim="longitude",
    y_dim="latitude",
    method="mean",
    chunks_per_shard=4,
)

print("Dataset tree:")
print(pyramid.dt)
print("\nEncoding:")
print(pyramid.encoding)

## 4. Write the pyramid to a new Icechunk store on R2

The pyramid contains **materialised** chunk data (not virtual references), so it lives
in its own repository.  `consolidated=False` is required — Icechunk manages its own
metadata index.

In [ ]:
pyr_storage = icechunk.s3_storage(
    bucket=r2_bucket,
    prefix="ndvi-cdr-pyramid",
    from_env=True,
    endpoint_url=r2_endpoint,
    region="auto",
    force_path_style=False,
)

pyr_repo    = icechunk.Repository.open_or_create(pyr_storage)
pyr_session = pyr_repo.writable_session("main")

pyramid.dt.to_zarr(
    pyr_session.store,
    mode="w",
    encoding=pyramid.encoding,
    consolidated=False,
)

snapshot_id = pyr_session.commit("Add GeoZarr multiscale pyramid — NDVI CDR Jan 2000")
print("Committed:", snapshot_id)

## 5. Verify — read back and inspect

Open the pyramid store and confirm the multiscales metadata and resolution levels.

In [ ]:
pyr_repo2    = icechunk.Repository.open(pyr_storage)
pyr_session2 = pyr_repo2.readonly_session("main")

import zarr
root = zarr.open_group(pyr_session2.store, mode="r")
print(root.tree())
print("\nmultiscales metadata:")
print(root.attrs.get("multiscales", root.attrs))

In [ ]:
import hvplot.xarray  # noqa

# Spot-check: read level 3 (coarsest) and plot — should be fast
ds_coarse = xr.open_zarr(pyr_session2.store, group="3", consolidated=False)

ds_coarse["NDVI"].isel(time=0).hvplot(
    rasterize=True, geo=True, global_extent=True,
    x="longitude", y="latitude", tiles="OSM",
    cmap="YlGn", clim=(-0.1, 1.0),
    title="AVHRR NDVI — 2000-01-01 (level 3, 8× coarsened)",
    width=800, height=400,
)